# GRUPO 06: Caso Electricidad

## Detección de Fraude Eléctrico 

### Características Principales:

1. **Prevención de Data Leakage**: División train-test como primer paso
2. **Optimización de Rendimiento**: Eliminación de iterrows(), uso de vectorización completa
3. **Pipeline Estructurado**: sklearn.pipeline y imblearn.pipeline para reproducibilidad
4. **Series Temporales**: Ordenamiento cronológico antes de interpolación
5. **Experimentación Sistemática**: Comparación de múltiples configuraciones
6. **Evaluación Metodológica**: Logging detallado y métricas comprensivas

### Técnicas Implementadas:

**INTERPOLACIÓN (2 métodos):**
- Interpolación Lineal
- Interpolación Polinomial (grado 2)

**BALANCEO DE DATOS (2 técnicas):**
- SMOTETomek (Over-sampling + Under-sampling)
- RandomUnderSampler (Under-sampling)

**DETECCIÓN DE OUTLIERS (2 técnicas):**
- Isolation Forest
- Local Outlier Factor (LOF)

### Arquitectura Técnica:
- Pipeline MLOps: sklearn.pipeline.Pipeline + imblearn.pipeline.Pipeline
- Series Temporales: Ordenamiento cronológico + interpolación vectorizada
- Detección de Outliers: Transformadores personalizados con sklearn API
- Experimentación: 4 configuraciones x 10 pipelines = 40 experimentos
- Evaluación: Métricas detalladas, matrices de confusión y análisis comparativo

In [11]:
# Importación de librerías
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento y pipelines
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# Balanceo de datos
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler

# Detección de outliers
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Modelos de clasificación
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# Métricas de evaluación
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, balanced_accuracy_score
)

# Configuración
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
np.random.seed(42)

print("Librerías importadas")
print("Configuración de reproducibilidad establecida (seed=42)")
print("Nuevas librerías: RandomUnderSampler, IsolationForest, LocalOutlierFactor")

Librerías importadas
Configuración de reproducibilidad establecida (seed=42)
Nuevas librerías: RandomUnderSampler, IsolationForest, LocalOutlierFactor


In [12]:
## 1. Carga y División Train-Test

# División como primer paso para prevenir data leakage
print("="*60)
print("PASO 1: CARGA Y DIVISIÓN TRAIN-TEST")
print("="*60)

# Carga de datos
df = pd.read_csv('Paper Electricidad/data.csv')
print(f"Datos cargados: {df.shape}")

# Identificación de columnas
id_col = 'CONS_NO'
target_col = 'FLAG' 
date_cols = [col for col in df.columns if col not in [id_col, target_col]]

print(f"\nColumnas identificadas:")
print(f"   - ID: {id_col}")  
print(f"   - Target: {target_col}")
print(f"   - Columnas de fechas: {len(date_cols)} columnas")
print(f"   - Rango de fechas: {date_cols[0]} a {date_cols[-1]}")

# Distribución original de clases
print(f"\nDistribución original de FLAG:")
flag_dist = df[target_col].value_counts(normalize=True)
print(f"   - Clase 0 (No fraude): {flag_dist[0]:.3f} ({df[target_col].value_counts()[0]:,} muestras)")
print(f"   - Clase 1 (Fraude): {flag_dist[1]:.3f} ({df[target_col].value_counts()[1]:,} muestras)")

# División train-test estratificada
print(f"\nDivisión train-test estratificada...")
X = df[date_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42, 
    stratify=y
)

print(f"   - X_train shape: {X_train.shape}")
print(f"   - X_test shape: {X_test.shape}")
print(f"   - y_train shape: {y_train.shape}")
print(f"   - y_test shape: {y_test.shape}")

# Verificación de distribuciones
print(f"\nDistribución en conjuntos divididos:")
train_dist = y_train.value_counts(normalize=True)
test_dist = y_test.value_counts(normalize=True)

print(f"   Train: Clase 0: {train_dist[0]:.3f}, Clase 1: {train_dist[1]:.3f}")
print(f"   Test:  Clase 0: {test_dist[0]:.3f}, Clase 1: {test_dist[1]:.3f}")
print(f"   Estratificación correcta: distribuciones preservadas")

print("\n" + "="*60)


PASO 1: CARGA Y DIVISIÓN TRAIN-TEST
Datos cargados: (42372, 1036)

Columnas identificadas:
   - ID: CONS_NO
   - Target: FLAG
   - Columnas de fechas: 1034 columnas
   - Rango de fechas: 2014/1/1 a 2016/9/9

Distribución original de FLAG:
   - Clase 0 (No fraude): 0.915 (38,757 muestras)
   - Clase 1 (Fraude): 0.085 (3,615 muestras)

División train-test estratificada...
Datos cargados: (42372, 1036)

Columnas identificadas:
   - ID: CONS_NO
   - Target: FLAG
   - Columnas de fechas: 1034 columnas
   - Rango de fechas: 2014/1/1 a 2016/9/9

Distribución original de FLAG:
   - Clase 0 (No fraude): 0.915 (38,757 muestras)
   - Clase 1 (Fraude): 0.085 (3,615 muestras)

División train-test estratificada...
   - X_train shape: (29660, 1034)
   - X_test shape: (12712, 1034)
   - y_train shape: (29660,)
   - y_test shape: (12712,)

Distribución en conjuntos divididos:
   Train: Clase 0: 0.915, Clase 1: 0.085
   Test:  Clase 0: 0.915, Clase 1: 0.085
   Estratificación correcta: distribuciones pr

## 2. TimeSeriesTransformer Personalizado

Usamos operaciones vectorizadas


In [ ]:
class OutlierDetector(BaseEstimator, TransformerMixin):
    """
    Detector de outliers que implementa 2 técnicas:
    1. Isolation Forest
    2. Local Outlier Factor (LOF)
    
    Agrega características de detección de outliers como nuevas columnas.
    Cumple con sklearn API para fit/transform.
    
    Parámetros:
    -----------
    methods : list, default=['isolation_forest', 'lof']
        Lista de métodos a usar: 'isolation_forest' y/o 'lof'
    contamination : float, default=0.1
        Proporción esperada de outliers en los datos
    """
    
    def __init__(self, methods=['isolation_forest', 'lof'], contamination=0.1):
        self.methods = methods
        self.contamination = contamination
        self.detectors_ = {}
        
    def fit(self, X, y=None):
        """
        Entrena los detectores de outliers solo en X_train.
        Previene data leakage.
        """
        print(f"   OutlierDetector.fit() - Métodos: {self.methods}")
        
        # Convertir a numpy array si es DataFrame
        X_array = X.values if isinstance(X, pd.DataFrame) else X
        
        if 'isolation_forest' in self.methods:
            print("   Entrenando Isolation Forest...")
            self.detectors_['IF'] = IsolationForest(
                contamination=self.contamination,
                random_state=42,
                n_jobs=-1
            )
            self.detectors_['IF'].fit(X_array)
            
        if 'lof' in self.methods:
            print("   Entrenando Local Outlier Factor...")
            self.detectors_['LOF'] = LocalOutlierFactor(
                n_neighbors=20,
                contamination=self.contamination,
                novelty=True,  # Permite predict en nuevos datos
                n_jobs=-1
            )
            self.detectors_['LOF'].fit(X_array)
            
        return self
    
    def transform(self, X):
        """
        Aplica detección de outliers y agrega características.
        Se aplica tanto a X_train como X_test.
        """
        print(f"   OutlierDetector.transform() - Detectando outliers en {X.shape[0]} muestras...")
        
        # Convertir a numpy array si es DataFrame
        X_array = X.values if isinstance(X, pd.DataFrame) else X
        
        # Copiar datos originales
        if isinstance(X, pd.DataFrame):
            X_transformed = X.copy()
        else:
            X_transformed = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
        
        # Isolation Forest
        if 'IF' in self.detectors_:
            outlier_labels_if = self.detectors_['IF'].predict(X_array)
            outlier_scores_if = self.detectors_['IF'].score_samples(X_array)
            
            X_transformed['is_outlier_IF'] = (outlier_labels_if == -1).astype(int)
            X_transformed['outlier_score_IF'] = -outlier_scores_if  # Invertir para mayor = más outlier
            
            n_outliers_if = (outlier_labels_if == -1).sum()
            print(f"   Isolation Forest detectó {n_outliers_if} outliers ({n_outliers_if/len(X)*100:.2f}%)")
            
        # Local Outlier Factor
        if 'LOF' in self.detectors_:
            outlier_labels_lof = self.detectors_['LOF'].predict(X_array)
            outlier_scores_lof = self.detectors_['LOF'].score_samples(X_array)
            
            X_transformed['is_outlier_LOF'] = (outlier_labels_lof == -1).astype(int)
            X_transformed['outlier_score_LOF'] = -outlier_scores_lof  # Invertir para mayor = más outlier
            
            n_outliers_lof = (outlier_labels_lof == -1).sum()
            print(f"   LOF detectó {n_outliers_lof} outliers ({n_outliers_lof/len(X)*100:.2f}%)")
            
        print(f"   Características agregadas: {X_transformed.shape[1] - X.shape[1]} nuevas columnas")
        return X_transformed

print("OutlierDetector definido (vectorizado)")
print("Implementa 2 técnicas: Isolation Forest y Local Outlier Factor")

OutlierDetector definido (vectorizado)
Implementa 2 técnicas: Isolation Forest y Local Outlier Factor
Cumple con sklearn API y previene data leakage


In [ ]:
class TimeSeriesTransformer(BaseEstimator, TransformerMixin):
    """
    Transformador personalizado para series temporales que:
    1. Ordena columnas cronológicamente
    2. Interpola valores faltantes vectorizadamente (Linear o Polynomial)
    3. Calcula características estadísticas vectorizadamente
    4. Cumple con sklearn API para fit/transform
    
    Parámetros:
    -----------
    interpolation_method : str, default='linear'
        Método de interpolación: 'linear' o 'polynomial'
    """
    
    def __init__(self, interpolation_method='linear'):
        self.interpolation_method = interpolation_method
        self.date_cols_sorted_ = None
        
    def _sort_date_columns(self, date_cols):
        """Ordena las columnas de fecha cronológicamente"""
        try:
            date_objects = pd.to_datetime(date_cols, format='%Y/%m/%d')
            sorted_indices = date_objects.argsort()
            return [date_cols[i] for i in sorted_indices]
        except:
            return date_cols
    
    def fit(self, X, y=None):
        """
        Aprende el ordenamiento cronológico de las columnas.
        Solo se llama con X_train para prevenir data leakage.
        """
        print(f"   TimeSeriesTransformer.fit() - Método: {self.interpolation_method}")
        
        self.date_cols_sorted_ = self._sort_date_columns(X.columns.tolist())
        
        print(f"   Columnas reordenadas cronológicamente: {len(self.date_cols_sorted_)} columnas")
        print(f"   Rango: {self.date_cols_sorted_[0]} -> {self.date_cols_sorted_[-1]}")
        
        return self
    
    def transform(self, X):
        """
        Transforma las series temporales de forma vectorizada.
        Se aplica tanto a X_train como X_test usando parámetros aprendidos.
        """
        print(f"   TimeSeriesTransformer.transform() - Procesando {X.shape[0]} muestras...")
        
        # Reordenar DataFrame según orden cronológico aprendido
        X_sorted = X[self.date_cols_sorted_].copy()
        
        # Interpolación vectorizada según método seleccionado
        print(f"   Aplicando interpolación {self.interpolation_method} vectorizada...")
        if self.interpolation_method == 'linear':
            X_interpolated = X_sorted.interpolate(
                method='linear', 
                axis=1,
                limit_direction='both'
            )
        elif self.interpolation_method == 'polynomial':
            # Interpolación polinomial de grado 2 (cuadrática)
            X_interpolated = X_sorted.copy()
            
            # Interpolar cada fila con polinomio de grado 2
            for idx in X_interpolated.index:
                row = X_interpolated.loc[idx]
                
                # Crear serie con índice numérico para interpolación polinomial
                row_numeric = pd.Series(row.values, index=range(len(row)))
                row_interpolated = row_numeric.interpolate(
                    method='polynomial',
                    order=2,  # Polinomio de grado 2 (cuadrática)
                    limit_direction='both'
                )
                X_interpolated.loc[idx] = row_interpolated.values
        else:
            raise ValueError(f"Método no soportado: {self.interpolation_method}")
        
        # Rellenar valores restantes
        X_interpolated = X_interpolated.fillna(method='ffill', axis=1)
        X_interpolated = X_interpolated.fillna(method='bfill', axis=1)
        X_interpolated = X_interpolated.fillna(0)
        
        # Calcular características estadísticas vectorizadamente
        print("   Calculando características estadísticas vectorizadas...")
        
        features = pd.DataFrame(index=X.index)
        
        # Características básicas
        features['mean'] = X_interpolated.mean(axis=1)
        features['std'] = X_interpolated.std(axis=1)
        features['min'] = X_interpolated.min(axis=1)
        features['max'] = X_interpolated.max(axis=1)
        features['median'] = X_interpolated.median(axis=1)
        
        # Características avanzadas
        features['skew'] = X_interpolated.skew(axis=1)
        features['kurtosis'] = X_interpolated.kurtosis(axis=1)
        
        # Percentiles
        features['q25'] = X_interpolated.quantile(0.25, axis=1)
        features['q75'] = X_interpolated.quantile(0.75, axis=1)
        
        # Características de conteo vectorizadas
        features['zero_count'] = (X_interpolated == 0).sum(axis=1)
        features['total_consumption'] = X_interpolated.sum(axis=1)
        
        # Características derivadas
        features['range'] = features['max'] - features['min']
        features['iqr'] = features['q75'] - features['q25']
        features['cv'] = features['std'] / (features['mean'] + 1e-8)
        
        print(f"   Características extraídas: {features.shape}")
        
        return features

print("TimeSeriesTransformer definido")
print("Soporta 2 métodos de interpolación: 'linear' y 'polynomial'")
print("Polynomial usa grado 2 (cuadrática)")

TimeSeriesTransformer definido (vectorizado, sin iterrows)
Soporta 2 métodos de interpolación: 'linear' y 'polynomial'
Polynomial usa grado 2 (cuadrática) con fallback a linear
Cumple con sklearn API: fit() aprende, transform() aplica
Prevención de data leakage: fit() solo en train, transform() en ambos


## 3. Pipeline de Preprocesamiento


In [15]:
# Definir múltiples configuraciones de preprocesamiento
print("="*60)
print("PASO 2: CREACIÓN DE PIPELINES DE PREPROCESAMIENTO")
print("="*60)

# Crear diccionario para almacenar todas las configuraciones procesadas
preprocessing_configs = {}

# Configuración 1: Linear + Sin Outliers
print("\n[CONFIG 1/4] Linear Interpolation + Sin Detección de Outliers")
print("-" * 60)
ts_linear = TimeSeriesTransformer(interpolation_method='linear')
preprocessor_linear = ColumnTransformer(
    transformers=[('timeseries', ts_linear, date_cols)],
    remainder='drop'
)

print("Ajustando preprocessor en X_train...")
preprocessor_linear.fit(X_train, y_train)

print("Transformando conjuntos...")
X_train_linear = preprocessor_linear.transform(X_train)
X_test_linear = preprocessor_linear.transform(X_test)

feature_names_base = ['mean', 'std', 'min', 'max', 'median', 'skew', 'kurtosis', 
                      'q25', 'q75', 'zero_count', 'total_consumption', 'range', 'iqr', 'cv']

X_train_linear = pd.DataFrame(X_train_linear, index=X_train.index, columns=feature_names_base)
X_test_linear = pd.DataFrame(X_test_linear, index=X_test.index, columns=feature_names_base)

preprocessing_configs['Linear_NoOutlier'] = {
    'X_train': X_train_linear,
    'X_test': X_test_linear,
    'description': 'Interpolación Lineal sin Detección de Outliers'
}

print(f"Shape: X_train={X_train_linear.shape}, X_test={X_test_linear.shape}")

# Configuración 2: Linear + Con Outliers
print("\n[CONFIG 2/4] Linear Interpolation + Detección de Outliers (IF + LOF)")
print("-" * 60)
outlier_detector = OutlierDetector(methods=['isolation_forest', 'lof'], contamination=0.1)

print("Entrenando detector de outliers en X_train...")
outlier_detector.fit(X_train_linear, y_train)

print("Aplicando detección de outliers...")
X_train_linear_outlier = outlier_detector.transform(X_train_linear)
X_test_linear_outlier = outlier_detector.transform(X_test_linear)

preprocessing_configs['Linear_Outlier'] = {
    'X_train': X_train_linear_outlier,
    'X_test': X_test_linear_outlier,
    'description': 'Interpolación Lineal con Detección de Outliers (IF + LOF)'
}

print(f"Shape: X_train={X_train_linear_outlier.shape}, X_test={X_test_linear_outlier.shape}")

# Configuración 3: Polynomial + Sin Outliers
print("\n[CONFIG 3/4] Polynomial Interpolation + Sin Detección de Outliers")
print("-" * 60)
ts_poly = TimeSeriesTransformer(interpolation_method='polynomial')
preprocessor_poly = ColumnTransformer(
    transformers=[('timeseries', ts_poly, date_cols)],
    remainder='drop'
)

print("Ajustando preprocessor en X_train...")
preprocessor_poly.fit(X_train, y_train)

print("Transformando conjuntos...")
X_train_poly = preprocessor_poly.transform(X_train)
X_test_poly = preprocessor_poly.transform(X_test)

X_train_poly = pd.DataFrame(X_train_poly, index=X_train.index, columns=feature_names_base)
X_test_poly = pd.DataFrame(X_test_poly, index=X_test.index, columns=feature_names_base)

preprocessing_configs['Poly_NoOutlier'] = {
    'X_train': X_train_poly,
    'X_test': X_test_poly,
    'description': 'Interpolación Polinomial (grado 2) sin Detección de Outliers'
}

print(f"Shape: X_train={X_train_poly.shape}, X_test={X_test_poly.shape}")

# Configuración 4: Polynomial + Con Outliers
print("\n[CONFIG 4/4] Polynomial Interpolation + Detección de Outliers (IF + LOF)")
print("-" * 60)
outlier_detector_poly = OutlierDetector(methods=['isolation_forest', 'lof'], contamination=0.1)

print("Entrenando detector de outliers en X_train...")
outlier_detector_poly.fit(X_train_poly, y_train)

print("Aplicando detección de outliers...")
X_train_poly_outlier = outlier_detector_poly.transform(X_train_poly)
X_test_poly_outlier = outlier_detector_poly.transform(X_test_poly)

preprocessing_configs['Poly_Outlier'] = {
    'X_train': X_train_poly_outlier,
    'X_test': X_test_poly_outlier,
    'description': 'Interpolación Polinomial (grado 2) con Detección de Outliers (IF + LOF)'
}

print(f"Shape: X_train={X_train_poly_outlier.shape}, X_test={X_test_poly_outlier.shape}")

print("\n" + "="*60)
print("RESUMEN DE CONFIGURACIONES DE PREPROCESAMIENTO")
print("="*60)
for config_name, config_data in preprocessing_configs.items():
    print(f"{config_name}: {config_data['description']}")
    print(f"   Features: {config_data['X_train'].shape[1]}")

print("\n" + "="*60)

PASO 2: CREACIÓN DE PIPELINES DE PREPROCESAMIENTO

[CONFIG 1/4] Linear Interpolation + Sin Detección de Outliers
------------------------------------------------------------
Ajustando preprocessor en X_train...
   TimeSeriesTransformer.fit() - Método: linear
   Columnas reordenadas cronológicamente: 1034 columnas
   Rango: 2014/1/1 -> 2016/10/31
   TimeSeriesTransformer.transform() - Procesando 29660 muestras...
   Aplicando interpolación linear vectorizada...
   Calculando características estadísticas vectorizadas...
   Calculando características estadísticas vectorizadas...
   Características extraídas: (29660, 14)
Transformando conjuntos...
   TimeSeriesTransformer.transform() - Procesando 29660 muestras...
   Aplicando interpolación linear vectorizada...
   Características extraídas: (29660, 14)
Transformando conjuntos...
   TimeSeriesTransformer.transform() - Procesando 29660 muestras...
   Aplicando interpolación linear vectorizada...
   Calculando características estadísticas ve

## 4. Meta-Modelo Random Forest para Extracción de Características


In [16]:
# Meta-modelo Random Forest para generar características adicionales
print("="*60)
print("PASO 3: META-MODELO RANDOM FOREST")
print("="*60)

print("Generando RF_Anomaly_Score para cada configuración...")
print("Objetivo: Generar puntuación de anomalía para detección de patrones de fraude")

# Agregar RF_Anomaly_Score a cada configuración
for config_name, config_data in preprocessing_configs.items():
    print(f"\nProcesando configuración: {config_name}")
    print("-" * 50)
    
    X_train_config = config_data['X_train']
    X_test_config = config_data['X_test']
    
    # Entrenar Random Forest solo en X_train_config (prevenir data leakage)
    rf_meta = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    
    print(f"   Ajustando Random Forest en train ({X_train_config.shape})...")
    rf_meta.fit(X_train_config, y_train)
    
    # Generar probabilidades de fraude (RF_Anomaly_Score)
    print("   Generando RF_Anomaly_Score...")
    rf_train_proba = rf_meta.predict_proba(X_train_config)[:, 1]
    rf_test_proba = rf_meta.predict_proba(X_test_config)[:, 1]
    
    # Agregar nueva característica
    X_train_config['RF_Anomaly_Score'] = rf_train_proba
    X_test_config['RF_Anomaly_Score'] = rf_test_proba
    
    print(f"   Train: min={rf_train_proba.min():.4f}, max={rf_train_proba.max():.4f}, mean={rf_train_proba.mean():.4f}")
    print(f"   Test:  min={rf_test_proba.min():.4f}, max={rf_test_proba.max():.4f}, mean={rf_test_proba.mean():.4f}")
    print(f"   Shape final: {X_train_config.shape}")

print("\n" + "="*60)
print("META-MODELO RANDOM FOREST COMPLETADO PARA TODAS LAS CONFIGURACIONES")
print("="*60)

PASO 3: META-MODELO RANDOM FOREST
Generando RF_Anomaly_Score para cada configuración...
Objetivo: Generar puntuación de anomalía para detección de patrones de fraude

Procesando configuración: Linear_NoOutlier
--------------------------------------------------
   Ajustando Random Forest en train ((29660, 14))...
   Generando RF_Anomaly_Score...
   Train: min=0.0012, max=0.9840, mean=0.0845
   Test:  min=0.0012, max=0.9634, mean=0.0841
   Shape final: (29660, 15)

Procesando configuración: Linear_Outlier
--------------------------------------------------
   Ajustando Random Forest en train ((29660, 18))...
   Generando RF_Anomaly_Score...
   Train: min=0.0012, max=0.9840, mean=0.0845
   Test:  min=0.0012, max=0.9634, mean=0.0841
   Shape final: (29660, 15)

Procesando configuración: Linear_Outlier
--------------------------------------------------
   Ajustando Random Forest en train ((29660, 18))...
   Generando RF_Anomaly_Score...
   Train: min=0.0002, max=0.9672, mean=0.0843
   Test: 

## 5. Pipelines de Modelos con Balanceo


In [17]:
# Definir pipelines con imblearn (balanceo + modelo)
print("="*60)
print("PASO 4: PIPELINES DE BALANCEO Y MODELADO")
print("="*60)

print("Definiendo pipelines imblearn (sampler + modelo)...")
print("2 técnicas de balanceo: SMOTETomek y RandomUnderSampler")

# Diccionario de pipelines con diferentes combinaciones
pipelines = {
    # Técnica 1: SMOTETomek (Over-sampling + Under-sampling)
    'SVM (SMOTETomek)': ImbPipeline([
        ('sampler', SMOTETomek(random_state=42)),
        ('model', SVC(C=1, probability=True, random_state=42))
    ]),
    'GNB (SMOTETomek)': ImbPipeline([
        ('sampler', SMOTETomek(random_state=42)),
        ('model', GaussianNB(var_smoothing=1e-8))
    ]),
    'RF (SMOTETomek)': ImbPipeline([
        ('sampler', SMOTETomek(random_state=42)),
        ('model', RandomForestClassifier(max_depth=10, n_estimators=100, random_state=42))
    ]),
    'LR (SMOTETomek)': ImbPipeline([
        ('sampler', SMOTETomek(random_state=42)),
        ('model', LogisticRegression(C=1, random_state=42, max_iter=1000))
    ]),
    
    # Técnica 2: RandomUnderSampler (Under-sampling)
    'SVM (UnderSampler)': ImbPipeline([
        ('sampler', RandomUnderSampler(random_state=42)),
        ('model', SVC(C=1, probability=True, random_state=42))
    ]),
    'GNB (UnderSampler)': ImbPipeline([
        ('sampler', RandomUnderSampler(random_state=42)),
        ('model', GaussianNB(var_smoothing=1e-8))
    ]),
    'RF (UnderSampler)': ImbPipeline([
        ('sampler', RandomUnderSampler(random_state=42)),
        ('model', RandomForestClassifier(max_depth=10, n_estimators=100, random_state=42))
    ]),
    'LR (UnderSampler)': ImbPipeline([
        ('sampler', RandomUnderSampler(random_state=42)),
        ('model', LogisticRegression(C=1, random_state=42, max_iter=1000))
    ]),
    
    # Sin balanceo para comparación
    'SVM (Sin Balanceo)': ImbPipeline([
        ('model', SVC(C=1, probability=True, random_state=42))
    ]),
    'GNB (Sin Balanceo)': ImbPipeline([
        ('model', GaussianNB(var_smoothing=1e-8))
    ])
}

print(f"\n{len(pipelines)} pipelines definidos:")
print("\nTécnica SMOTETomek:")
for name in [k for k in pipelines.keys() if 'SMOTETomek' in k]:
    print(f"   - {name}")

print("\nTécnica RandomUnderSampler:")
for name in [k for k in pipelines.keys() if 'UnderSampler' in k]:
    print(f"   - {name}")
    
print("\nSin Balanceo (baseline):")
for name in [k for k in pipelines.keys() if 'Sin Balanceo' in k]:
    print(f"   - {name}")

print("\n" + "="*60)
print("PIPELINES IMBLEARN LISTOS PARA ENTRENAMIENTO")
print("="*60)

PASO 4: PIPELINES DE BALANCEO Y MODELADO
Definiendo pipelines imblearn (sampler + modelo)...
2 técnicas de balanceo: SMOTETomek y RandomUnderSampler

10 pipelines definidos:

Técnica SMOTETomek:
   - SVM (SMOTETomek)
   - GNB (SMOTETomek)
   - RF (SMOTETomek)
   - LR (SMOTETomek)

Técnica RandomUnderSampler:
   - SVM (UnderSampler)
   - GNB (UnderSampler)
   - RF (UnderSampler)
   - LR (UnderSampler)

Sin Balanceo (baseline):
   - SVM (Sin Balanceo)
   - GNB (Sin Balanceo)

PIPELINES IMBLEARN LISTOS PARA ENTRENAMIENTO


## 6. Entrenamiento y Evaluación


In [18]:
# Entrenamiento y evaluación de todos los pipelines con todas las configuraciones
print("="*60)
print("PASO 5: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS")
print("="*60)

# Almacenar resultados de evaluación
results = []

print("Iniciando entrenamiento de pipelines...")
print(f"   Total de configuraciones: {len(preprocessing_configs)}")
print(f"   Total de pipelines: {len(pipelines)}")
print(f"   Total de experimentos: {len(preprocessing_configs) * len(pipelines)}")
print("   Métricas: Precision, Recall, F1-Score, ROC-AUC")
print()

# Iterar sobre cada configuración de preprocesamiento
for config_name, config_data in preprocessing_configs.items():
    print("="*80)
    print(f"CONFIGURACIÓN: {config_data['description']}")
    print("="*80)
    
    X_train_config = config_data['X_train']
    X_test_config = config_data['X_test']
    
    # Iterar sobre cada pipeline
    for pipeline_name, pipeline in pipelines.items():
        experiment_name = f"{config_name} | {pipeline_name}"
        print(f"\nEntrenando: {experiment_name}")
        print("-" * 70)
        
        # Entrenar pipeline (fit aplica balanceo solo en entrenamiento)
        print("   Ajustando pipeline (con balanceo si aplica)...")
        pipeline.fit(X_train_config, y_train)
        
        # Predecir en test (predict NO aplica balanceo)
        print("   Prediciendo en conjunto de test...")
        y_pred = pipeline.predict(X_test_config)
        
        # Obtener probabilidades si el modelo las soporta
        try:
            y_proba = pipeline.predict_proba(X_test_config)[:, 1]
            roc_auc = roc_auc_score(y_test, y_proba)
        except:
            y_proba = None
            roc_auc = None
        
        # Calcular métricas
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        accuracy = accuracy_score(y_test, y_pred)
        balanced_acc = balanced_accuracy_score(y_test, y_pred)
        
        print(f"   Accuracy: {accuracy:.4f} | Balanced Acc: {balanced_acc:.4f} | F1: {f1:.4f}", end="")
        if roc_auc is not None:
            print(f" | ROC-AUC: {roc_auc:.4f}")
        else:
            print()
        
        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred)
        
        # Almacenar resultados
        results.append({
            'config': config_name,
            'config_description': config_data['description'],
            'pipeline': pipeline_name,
            'experiment': experiment_name,
            'accuracy': accuracy,
            'balanced_accuracy': balanced_acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'roc_auc': roc_auc,
            'tn': cm[0,0],
            'fp': cm[0,1],
            'fn': cm[1,0],
            'tp': cm[1,1]
        })

print("\n" + "="*80)
print(f"EVALUACIÓN COMPLETADA: {len(results)} experimentos ")
print("="*80)

PASO 5: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS
Iniciando entrenamiento de pipelines...
   Total de configuraciones: 4
   Total de pipelines: 10
   Total de experimentos: 40
   Métricas: Precision, Recall, F1-Score, ROC-AUC

CONFIGURACIÓN: Interpolación Lineal sin Detección de Outliers

Entrenando: Linear_NoOutlier | SVM (SMOTETomek)
----------------------------------------------------------------------
   Ajustando pipeline (con balanceo si aplica)...
   Prediciendo en conjunto de test...
   Prediciendo en conjunto de test...
   Accuracy: 0.7303 | Balanced Acc: 0.5894 | F1: 0.7839 | ROC-AUC: 0.6410

Entrenando: Linear_NoOutlier | GNB (SMOTETomek)
----------------------------------------------------------------------
   Ajustando pipeline (con balanceo si aplica)...
   Accuracy: 0.7303 | Balanced Acc: 0.5894 | F1: 0.7839 | ROC-AUC: 0.6410

Entrenando: Linear_NoOutlier | GNB (SMOTETomek)
----------------------------------------------------------------------
   Ajustando pipeline (con bala

In [20]:
# Análisis comparativo de resultados
print("="*80)
print("ANÁLISIS COMPARATIVO DE RESULTADOS")
print("="*80)

# Crear DataFrame con resultados
results_df = pd.DataFrame(results)

# Ordenar por F1-Score
results_df_sorted = results_df.sort_values('f1_score', ascending=False)

print("\nTOP 10 MEJORES CONFIGURACIONES (por F1-Score):")
print("="*80)
print(f"{'Rank':<5} {'F1-Score':<10} {'ROC-AUC':<10} {'Balanced Acc':<13} {'Configuración':<35} {'Pipeline'}")
print("-"*80)

for idx, (i, row) in enumerate(results_df_sorted.head(10).iterrows(), 1):
    roc_str = f"{row['roc_auc']:.4f}" if row['roc_auc'] is not None else "N/A"
    config_short = row['config']
    pipeline_short = row['pipeline']
    print(f"{idx:<5} {row['f1_score']:<10.4f} {roc_str:<10} {row['balanced_accuracy']:<13.4f} {config_short:<35} {pipeline_short}")

# Identificar mejor modelo
best_model = results_df_sorted.iloc[0]
print(f"\n{'='*80}")
print("MEJOR MODELO GLOBAL")
print("="*80)
print(f"Configuración: {best_model['config_description']}")
print(f"Pipeline: {best_model['pipeline']}")
print(f"\nMétricas:")
print(f"   F1-Score: {best_model['f1_score']:.4f}")
print(f"   ROC-AUC: {best_model['roc_auc']:.4f}" if best_model['roc_auc'] is not None else "   ROC-AUC: N/A")
print(f"   Accuracy: {best_model['accuracy']:.4f}")
print(f"   Balanced Accuracy: {best_model['balanced_accuracy']:.4f}")
print(f"   Precision: {best_model['precision']:.4f}")
print(f"   Recall: {best_model['recall']:.4f}")
print(f"\nMatriz de Confusión:")
print(f"   TN={best_model['tn']}, FP={best_model['fp']}")
print(f"   FN={best_model['fn']}, TP={best_model['tp']}")

# Análisis por dimensión
print("\n" + "="*80)
print("ANÁLISIS POR DIMENSIONES")
print("="*80)

# Comparación por Interpolación
print("\n1. COMPARACIÓN POR MÉTODO DE INTERPOLACIÓN:")
print("-" * 60)
interp_comparison = results_df.groupby(results_df['config'].str.contains('Linear').map({True: 'Linear', False: 'Polynomial'})).agg({
    'f1_score': ['mean', 'std', 'max'],
    'roc_auc': ['mean', 'max'],
    'balanced_accuracy': ['mean', 'max']
}).round(4)
print(interp_comparison)

# Comparación por Detección de Outliers
print("\n2. COMPARACIÓN POR DETECCIÓN DE OUTLIERS:")
print("-" * 60)
outlier_comparison = results_df.groupby(results_df['config'].str.contains('Outlier').map({True: 'Con Outliers', False: 'Sin Outliers'})).agg({
    'f1_score': ['mean', 'std', 'max'],
    'roc_auc': ['mean', 'max'],
    'balanced_accuracy': ['mean', 'max']
}).round(4)
print(outlier_comparison)

# Comparación por Técnica de Balanceo
print("\n3. COMPARACIÓN POR TÉCNICA DE BALANCEO:")
print("-" * 60)
def get_balancing_method(pipeline_name):
    if 'SMOTETomek' in pipeline_name:
        return 'SMOTETomek'
    elif 'UnderSampler' in pipeline_name:
        return 'RandomUnderSampler'
    else:
        return 'Sin Balanceo'

results_df['balancing_method'] = results_df['pipeline'].apply(get_balancing_method)
balancing_comparison = results_df.groupby('balancing_method').agg({
    'f1_score': ['mean', 'std', 'max'],
    'roc_auc': ['mean', 'max'],
    'balanced_accuracy': ['mean', 'max']
}).round(4)
print(balancing_comparison)


print("\n" + "="*80)

ANÁLISIS COMPARATIVO DE RESULTADOS

TOP 10 MEJORES CONFIGURACIONES (por F1-Score):
Rank  F1-Score   ROC-AUC    Balanced Acc  Configuración                       Pipeline
--------------------------------------------------------------------------------
1     0.8822     0.6171     0.5304        Linear_NoOutlier                    GNB (SMOTETomek)
2     0.8817     0.6198     0.5268        Linear_NoOutlier                    GNB (Sin Balanceo)
3     0.8805     0.6043     0.5522        Poly_NoOutlier                      GNB (SMOTETomek)
4     0.8805     0.6043     0.5522        Poly_Outlier                        GNB (SMOTETomek)
5     0.8802     0.6004     0.5363        Poly_NoOutlier                      GNB (Sin Balanceo)
6     0.8802     0.6006     0.5363        Poly_Outlier                        GNB (Sin Balanceo)
7     0.8797     0.6084     0.5268        Linear_NoOutlier                    GNB (UnderSampler)
8     0.8796     0.6084     0.5268        Linear_Outlier                    

## 7. Resumen final


In [21]:
# Resumen final técnico
print("="*80)
print("RESUMEN FINAL - IMPLEMENTACIÓN ")
print("="*80)

print(f"   Configuración: {best_model['config_description']}")
print(f"   Pipeline: {best_model['pipeline']}")
print(f"   F1-Score: {best_model['f1_score']:.4f}")
print(f"   ROC-AUC: {best_model['roc_auc']:.4f}" if best_model['roc_auc'] is not None else "   ROC-AUC: N/A")
print(f"   Balanced Accuracy: {best_model['balanced_accuracy']:.4f}")


RESUMEN FINAL - IMPLEMENTACIÓN 
   Configuración: Interpolación Lineal sin Detección de Outliers
   Pipeline: GNB (SMOTETomek)
   F1-Score: 0.8822
   ROC-AUC: 0.6171
   Balanced Accuracy: 0.5304


## 8. Análisis Detallado de Resultados

In [22]:
# Análisis detallado por cada técnica implementada
print("="*80)

# 1. Análisis de Interpolación
print("\n1. ANÁLISIS DE MÉTODOS DE INTERPOLACIÓN")
print("-" * 80)

linear_results = results_df[results_df['config'].str.contains('Linear')]
poly_results = results_df[results_df['config'].str.contains('Poly')]

print(f"\nInterpolación Lineal:")
print(f"   Experimentos: {len(linear_results)}")
print(f"   F1-Score promedio: {linear_results['f1_score'].mean():.4f} (+/- {linear_results['f1_score'].std():.4f})")
print(f"   F1-Score máximo: {linear_results['f1_score'].max():.4f}")
print(f"   ROC-AUC promedio: {linear_results['roc_auc'].mean():.4f}")

print(f"\nInterpolación Polinomial (grado 2):")
print(f"   Experimentos: {len(poly_results)}")
print(f"   F1-Score promedio: {poly_results['f1_score'].mean():.4f} (+/- {poly_results['f1_score'].std():.4f})")
print(f"   F1-Score máximo: {poly_results['f1_score'].max():.4f}")
print(f"   ROC-AUC promedio: {poly_results['roc_auc'].mean():.4f}")

improvement_interp = ((poly_results['f1_score'].mean() - linear_results['f1_score'].mean()) / linear_results['f1_score'].mean()) * 100
print(f"\nMejora Polynomial vs Linear: {improvement_interp:+.2f}%")

# 2. Análisis de Detección de Outliers
print("\n" + "="*80)
print("2. ANÁLISIS DE DETECCIÓN DE OUTLIERS")
print("-" * 80)

no_outlier_results = results_df[results_df['config'].str.contains('NoOutlier')]
with_outlier_results = results_df[results_df['config'].str.contains('Outlier')]

print(f"\nSin Detección de Outliers:")
print(f"   Experimentos: {len(no_outlier_results)}")
print(f"   F1-Score promedio: {no_outlier_results['f1_score'].mean():.4f} (+/- {no_outlier_results['f1_score'].std():.4f})")
print(f"   F1-Score máximo: {no_outlier_results['f1_score'].max():.4f}")
print(f"   ROC-AUC promedio: {no_outlier_results['roc_auc'].mean():.4f}")

print(f"\nCon Detección de Outliers (IF + LOF):")
print(f"   Experimentos: {len(with_outlier_results)}")
print(f"   F1-Score promedio: {with_outlier_results['f1_score'].mean():.4f} (+/- {with_outlier_results['f1_score'].std():.4f})")
print(f"   F1-Score máximo: {with_outlier_results['f1_score'].max():.4f}")
print(f"   ROC-AUC promedio: {with_outlier_results['roc_auc'].mean():.4f}")

improvement_outlier = ((with_outlier_results['f1_score'].mean() - no_outlier_results['f1_score'].mean()) / no_outlier_results['f1_score'].mean()) * 100
print(f"\nMejora Con Outliers vs Sin Outliers: {improvement_outlier:+.2f}%")

# 3. Análisis de Técnicas de Balanceo
print("\n" + "="*80)
print("3. ANÁLISIS DE TÉCNICAS DE BALANCEO")
print("-" * 80)

smote_results = results_df[results_df['pipeline'].str.contains('SMOTETomek')]
under_results = results_df[results_df['pipeline'].str.contains('UnderSampler')]
no_balance_results = results_df[results_df['pipeline'].str.contains('Sin Balanceo')]

print(f"\nSMOTETomek (Over + Under Sampling):")
print(f"   Experimentos: {len(smote_results)}")
print(f"   F1-Score promedio: {smote_results['f1_score'].mean():.4f} (+/- {smote_results['f1_score'].std():.4f})")
print(f"   F1-Score máximo: {smote_results['f1_score'].max():.4f}")
print(f"   ROC-AUC promedio: {smote_results['roc_auc'].mean():.4f}")

print(f"\nRandomUnderSampler (Under Sampling):")
print(f"   Experimentos: {len(under_results)}")
print(f"   F1-Score promedio: {under_results['f1_score'].mean():.4f} (+/- {under_results['f1_score'].std():.4f})")
print(f"   F1-Score máximo: {under_results['f1_score'].max():.4f}")
print(f"   ROC-AUC promedio: {under_results['roc_auc'].mean():.4f}")

print(f"\nSin Balanceo (Baseline):")
print(f"   Experimentos: {len(no_balance_results)}")
print(f"   F1-Score promedio: {no_balance_results['f1_score'].mean():.4f} (+/- {no_balance_results['f1_score'].std():.4f})")
print(f"   F1-Score máximo: {no_balance_results['f1_score'].max():.4f}")
print(f"   ROC-AUC promedio: {no_balance_results['roc_auc'].mean():.4f}")

improvement_smote = ((smote_results['f1_score'].mean() - no_balance_results['f1_score'].mean()) / no_balance_results['f1_score'].mean()) * 100
improvement_under = ((under_results['f1_score'].mean() - no_balance_results['f1_score'].mean()) / no_balance_results['f1_score'].mean()) * 100

print(f"\nMejora SMOTETomek vs Sin Balanceo: {improvement_smote:+.2f}%")
print(f"Mejora RandomUnderSampler vs Sin Balanceo: {improvement_under:+.2f}%")

# 4. Tabla resumen de mejores configuraciones
print("\n" + "="*80)
print("4. MEJORES CONFIGURACIONES POR CATEGORÍA")
print("-" * 80)

print(f"\nMejor con Interpolación Lineal:")
best_linear = linear_results.nlargest(1, 'f1_score').iloc[0]
print(f"   {best_linear['experiment']}")
print(f"   F1-Score: {best_linear['f1_score']:.4f}, ROC-AUC: {best_linear['roc_auc']:.4f}")

print(f"\nMejor con Interpolación Polinomial:")
best_poly = poly_results.nlargest(1, 'f1_score').iloc[0]
print(f"   {best_poly['experiment']}")
print(f"   F1-Score: {best_poly['f1_score']:.4f}, ROC-AUC: {best_poly['roc_auc']:.4f}")

print(f"\nMejor con SMOTETomek:")
best_smote = smote_results.nlargest(1, 'f1_score').iloc[0]
print(f"   {best_smote['experiment']}")
print(f"   F1-Score: {best_smote['f1_score']:.4f}, ROC-AUC: {best_smote['roc_auc']:.4f}")

print(f"\nMejor con RandomUnderSampler:")
best_under = under_results.nlargest(1, 'f1_score').iloc[0]
print(f"   {best_under['experiment']}")
print(f"   F1-Score: {best_under['f1_score']:.4f}, ROC-AUC: {best_under['roc_auc']:.4f}")

print(f"\nMejor con Detección de Outliers:")
best_with_outlier = with_outlier_results.nlargest(1, 'f1_score').iloc[0]
print(f"   {best_with_outlier['experiment']}")
print(f"   F1-Score: {best_with_outlier['f1_score']:.4f}, ROC-AUC: {best_with_outlier['roc_auc']:.4f}")

print("\n" + "="*80)
print("="*80)


1. ANÁLISIS DE MÉTODOS DE INTERPOLACIÓN
--------------------------------------------------------------------------------

Interpolación Lineal:
   Experimentos: 20
   F1-Score promedio: 0.7077 (+/- 0.3021)
   F1-Score máximo: 0.8822
   ROC-AUC promedio: 0.6234

Interpolación Polinomial (grado 2):
   Experimentos: 20
   F1-Score promedio: 0.8184 (+/- 0.1228)
   F1-Score máximo: 0.8805
   ROC-AUC promedio: 0.6086

Mejora Polynomial vs Linear: +15.63%

2. ANÁLISIS DE DETECCIÓN DE OUTLIERS
--------------------------------------------------------------------------------

Sin Detección de Outliers:
   Experimentos: 20
   F1-Score promedio: 0.8305 (+/- 0.0881)
   F1-Score máximo: 0.8822
   ROC-AUC promedio: 0.6272

Con Detección de Outliers (IF + LOF):
   Experimentos: 40
   F1-Score promedio: 0.7631 (+/- 0.2344)
   F1-Score máximo: 0.8822
   ROC-AUC promedio: 0.6160

Mejora Con Outliers vs Sin Outliers: -8.13%

3. ANÁLISIS DE TÉCNICAS DE BALANCEO
--------------------------------------------

In [23]:
output_file = 'resultados_experimentacion_completa.csv'
results_df.to_csv(output_file, index=False)
print(f"Resultados exportados a: {output_file}")
print(f"Total de registros: {len(results_df)}")
print(f"\nColumnas exportadas:")
for col in results_df.columns:
    print(f"   - {col}")
    

Resultados exportados a: resultados_experimentacion_completa.csv
Total de registros: 40

Columnas exportadas:
   - config
   - config_description
   - pipeline
   - experiment
   - accuracy
   - balanced_accuracy
   - precision
   - recall
   - f1_score
   - roc_auc
   - tn
   - fp
   - fn
   - tp
   - balancing_method



### Configuraciones 

1. **Linear_NoOutlier**: Interpolación Lineal sin Detección de Outliers
2. **Linear_Outlier**: Interpolación Lineal con Detección de Outliers (IF + LOF)
3. **Poly_NoOutlier**: Interpolación Polinomial (grado 2) sin Detección de Outliers
4. **Poly_Outlier**: Interpolación Polinomial (grado 2) con Detección de Outliers (IF + LOF)

### Pipelines de Modelado

**Con Balanceo SMOTETomek:**
- SVM (SMOTETomek)
- GNB (SMOTETomek)
- RF (SMOTETomek)
- LR (SMOTETomek)

**Con Balanceo RandomUnderSampler:**
- SVM (UnderSampler)
- GNB (UnderSampler)
- RF (UnderSampler)
- LR (UnderSampler)

**Sin Balanceo (Baseline):**
- SVM (Sin Balanceo)
- GNB (Sin Balanceo)

### Total de Experimentos

- **Configuraciones de Preprocesamiento**: 4
- **Pipelines de Modelado**: 10
- **Total de Experimentos**: 40

Cada experimento es evaluado con métricas completas (Accuracy, Precision, Recall, F1-Score, ROC-AUC, Balanced Accuracy) permitiendo análisis comparativo robusto.